In [2]:
%pip install pymupdf ollama sentence-transformers pandas scipy scikit-learn pydantic --break-system-packages

  Using cached pymupdf-1.28.2-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (26 kB)
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached pymupdf-1.28.2-cp310-abi3-manylinux_2_28_x86_64.whl (25.8 MB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 3.9 MB/s  0:00:00
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.4-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7

In [8]:
import os
import json
import zipfile
import pymupdf 
from ollama import Client

In [27]:
ZIP_FILE_PATH = "../data/CV-data-scientist.zip"
EXTRACT_FOLDER = "../data/extracted_cvs"
OUTPUT_FILE = "../data/ground_truth.json"
client = Client(host="http://localhost:11434")

In [28]:
if not os.path.exists(EXTRACT_FOLDER):
    os.makedirs(EXTRACT_FOLDER)

print(f"Extracting {ZIP_FILE_PATH}...")
try:
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_FOLDER)
except FileNotFoundError:
    print(f"ERROR: Cannot find {ZIP_FILE_PATH}. Please check the file path.")
    raise

Extracting ../data/CV-data-scientist.zip...


In [29]:
cv_texts = []
for root, dirs, files in os.walk(EXTRACT_FOLDER):
    for filename in files:
        if filename.lower().endswith(".pdf") and not filename.startswith("._"):
            path = os.path.join(root, filename)
            try:
                doc = fitz.open(path)
                text = "\n".join([page.get_text() for page in doc])
                # Take the first 2000 chars to avoid overwhelming the context window
                cv_texts.append(f"--- Candidate: {filename} ---\n{text[:2000]}\n")
            except Exception as e:
                print(f"Could not read {filename}: {e}")

all_cv_text = "\n".join(cv_texts)
print(f"Successfully read {len(cv_texts)} PDFs.")

Successfully read 11 PDFs.


In [31]:
import subprocess

result = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True
)

print(result.stdout)

NAME                                                            ID              SIZE      MODIFIED     
qwen3:0.6b                                                      7df6b6e09427    522 MB    6 days ago      
bge-m3:latest                                                   790764642607    1.2 GB    6 days ago      
qwen3:1.7b                                                      3b6a3cd17b97    1.4 GB    3 weeks ago     
qwen3:4b-instruct                                               e57fe8d88e66    2.5 GB    5 weeks ago     
qwen3:14b                                                       bdbd181c33f2    9.3 GB    6 weeks ago     
Qwen3-4B-Instruct-2507-SOAP-subash-GGUF:latest                  4d0d3b2498dc    2.5 GB    2 months ago    
hf.co/rakuzai/Qwen3-4B-Instruct-2507-SOAP-subash-GGUF:latest    555c8b38b624    2.5 GB    3 months ago    
gpt-oss:20b                                                     a554d89b0cb2    13 GB     4 months ago    
devstral-2:123b-cloud                   

In [73]:
candidate_list = "\n".join(
    f"- {filename}"
    for filename in pdf_files
)

system_prompt = f"""You are an expert technical recruiter screening candidates for a JUNIOR AI / MACHINE LEARNING / DATA SCIENCE position.

Your task is to rank exactly 11 candidates from 1 (BEST) to 11 (WORST) based on their actual suitability for this role.

The target candidate is a junior-level person with approximately 0–2 years of RELEVANT AI/ML/Data Science experience. Internships, academic projects, research, freelance work, and substantial personal projects can count as relevant experience.

Some CVs are written in English and some are written in Indonesian. Understand and evaluate both languages equally.

==================================================
CANDIDATES
==================================================

The exact candidate filenames are:

{candidate_list}

Use these filenames EXACTLY in the final JSON.

==================================================
CORE OBJECTIVE & RELEVANCE GATE
==================================================

The most important question is:
"Does this candidate actually have relevant AI, Machine Learning, Data Science, Data Analytics, or statistical modeling experience?"

A candidate should be ranked highly ONLY if their CV provides genuine evidence that they have worked with data, AI, machine learning, analytics, or statistical modeling.

STRONG RELEVANCE: ML/DS employment, ML/DS internships, ML research, substantial ML projects.
MODERATE RELEVANCE: Minor analytics work, some academic ML exposure.
WEAK/NO RELEVANCE: IT Support, Networking, Web Development, Administration, General Software Engineering without ML context.

==================================================
STRICT PENALTIES & DISQUALIFIERS (DO NOT IGNORE)
==================================================

To prevent ranking irrelevant candidates highly, you MUST apply these strict penalties:

PENALTY 1: THE SENIORITY ILLUSION
If a candidate has many years of experience (e.g., 3+ years) but that experience is in IT Support, Networking, Hardware, or Backend Development with NO explicit Data Science projects, they MUST be ranked at the bottom (Rank 9, 10, or 11). Do not rank them #1 just because they have a long work history.

PENALTY 2: KEYWORD STUFFING
If a candidate simply lists "Machine Learning", "AI", or "Python" in a skills section, but their actual work history descriptions do not describe them building models or analyzing data, treat their ML experience as ZERO.

PENALTY 3: DEGREE WITHOUT EXPERIENCE
A Computer Science or Informatics degree does NOT automatically make someone an AI/ML candidate. If they have a degree but zero ML projects/internships, they must rank below candidates who have actual ML projects.

==================================================
JUNIOR LEVEL EXPECTATIONS
==================================================

This is a JUNIOR position.
- Around 0–2 years of relevant experience is appropriate.
- Strong internships and substantial academic/personal projects OVERRIDE long, irrelevant professional experience.
- Do not heavily penalize a lack of advanced MLOps or Cloud architecture.

==================================================
RANKING PRIORITY HIERARCHY
==================================================

Use this hierarchy:
1. RELEVANCE TO AI/ML/DATA SCIENCE (Highest weight)
2. QUALITY OF RELEVANT ML/DS PROJECTS
3. PRACTICAL PYTHON/DATA SKILLS
4. EDUCATION
5. GENERAL TECHNICAL EXPERIENCE (Lowest weight)

==================================================
OUTPUT REQUIREMENTS
==================================================

There are exactly 11 candidates.
Return exactly 11 entries.
Every candidate must appear exactly once.
Use every rank exactly once: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11.
Rank 1 = BEST candidate. Rank 11 = WORST candidate.
The JSON keys MUST be the exact filenames provided above.

Return ONLY a valid JSON dictionary.
No markdown. No explanations. No reasoning. No comments.

Example structure:
{{"exact_filename.pdf": 1}}
"""

In [74]:
print("Analyzing CVs and generating rankings... please wait.")
print("Length of CV text:", len(all_cv_text))
print(all_cv_text)
response = client.chat(
    model="qwen3:14b", 
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": all_cv_text}
    ],
    format="json",
    options={"temperature": 0.0}
)

Analyzing CVs and generating rankings... please wait.
Length of CV text: 21886
--- Candidate: AGUS RIYADI - agus riyadi.pdf ---
 
AGUS RIYADI 
 
KONTAK 
081804708422 
Agusr381@gmail.com 
Surakarta, Indonesia 
 
PENDIDIKAN 
 
 
STMIK AUB Surakarta 
Sistem Informasi 
2015 - 2019 
 
SMK 
Muhammadiyah 
6 
Karanganyar 
Teknik Komputer & Jaringan 
2012 - 2015 
KEAHLIAN 
• 
Hardware & Software 
• 
Bahasa Pemograman 
• 
Database 
• 
Microsoft (Office, Visio, 
Acces, 365) 
• 
Diagram 
(DFD, 
Flowchart) 
 
KEMAMPUAN 
• 
Komunikasi 
• 
Manajemen Proyek 
• 
Kolaboratif 
• 
Kerja Tim 
Lulusan ilmu komputer dengan jurusan Sistem Informasi dari 
salah satu perguruan tinggi swasta di Surakarta.Saya mempunyai 
minat dalam menganalisa dan memperhatikan hal-hal detail dalam 
memecahkan masalah.Dan mempunyai tekad yang kuat untuk 
berkembang serta berdedikasi tinggi dalam memecahkan 
masalah. 
 
PENGALAMAN KERJA 
 
Admin Logistik 
Toko buku Al Atsari (Maret 2023 – Oktober 2024) 
 
• 
Melakukan Pemantauan 

In [75]:
try:
    ground_truth = json.loads(response["message"]["content"])

    # Validate number of candidates
    if len(ground_truth) != 11:
        raise ValueError(
            f"Expected 11 candidates, but model returned {len(ground_truth)}"
        )

    # Validate ranks
    ranks = list(ground_truth.values())

    if sorted(ranks) != list(range(1, 12)):
        raise ValueError(
            f"Invalid rankings. Expected ranks 1-11 exactly once, got: {sorted(ranks)}"
        )

    # Validate filenames against your actual PDFs
    expected_files = set(pdf_files)  # whatever variable contains your 11 PDF filenames
    returned_files = set(ground_truth.keys())

    missing = expected_files - returned_files
    extra = returned_files - expected_files

    if missing:
        raise ValueError(f"Missing candidates: {missing}")

    if extra:
        raise ValueError(f"Unknown candidates: {extra}")

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(OUTPUT_FILE, "w") as f:
        json.dump(ground_truth, f, indent=4)

    print("✅ Valid ranking generated!")
    print(json.dumps(ground_truth, indent=2))

except Exception as e:
    print("❌ Invalid LLM output:")
    print(e)
    print("\nRaw output from LLM:")
    print(response["message"]["content"])

✅ Valid ranking generated!
{
  "AGUS RIYADI - agus riyadi.pdf": 6,
  "CV Ryana Purwaningrum_New - Ryana Purwaningrum.pdf": 11,
  "CV Dewi Permata by Naevaweb.pdf": 10,
  "CV - Nugrahita Purbasantika Paramashintaa - Nugrahita P.pdf": 9,
  "Agung Widiyanto - CV - Agung Widiyanto.pdf": 8,
  "cv_zahrazulhulaifahh.docx - Zahra Zul Hulaifah.pdf": 7,
  "CV_DS_Safina_Newest - Safina Nanda.pdf": 5,
  "CV Muhammad Abdul Latief 852025 - Latief Abdul.pdf": 4,
  "CV_Abdul Jabbar Robbani_ATS_Indo - Abdul Jabbar Robbani.pdf": 2,
  "CV ADJIE HARI FAJAR - Adjie Hari Fajar.pdf": 3,
  "CV AHMAD ROPII-New (1) - Ahmad Ropii.pdf": 1
}


In [46]:
ranks = sorted(ground_truth.values())

print("Ranks:", ranks)
print("Missing ranks:", set(range(1, 12)) - set(ranks))

Ranks: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Missing ranks: set()


In [77]:
import json
import os

# Your current configuration paths
OUTPUT_FILE = "../data/ground_truth.json"

# The corrected manual ranking
correct_ground_truth = {
  "CV Ryana Purwaningrum_New - Ryana Purwaningrum.pdf": 1,       
  "CV_Abdul Jabbar Robbani_ATS_Indo - Abdul Jabbar Robbani.pdf": 2, 
  "CV ADJIE HARI FAJAR - Adjie Hari Fajar.pdf": 3,
  "CV Muhammad Abdul Latief 852025 - Latief Abdul.pdf": 4,
  "CV_DS_Safina_Newest - Safina Nanda.pdf": 5,
  "Agung Widiyanto - CV - Agung Widiyanto.pdf": 6,
  "AGUS RIYADI - agus riyadi.pdf": 7,
  "cv_zahrazulhulaifahh.docx - Zahra Zul Hulaifah.pdf": 8,
  "CV - Nugrahita Purbasantika Paramashintaa - Nugrahita P.pdf": 9,                         
  "CV AHMAD ROPII-New (1) - Ahmad Ropii.pdf": 10, 
  "CV Dewi Permata by Naevaweb.pdf": 11
}

# Ensure the data directory exists
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Save the corrected ground truth
with open(OUTPUT_FILE, "w") as f:
    json.dump(correct_ground_truth, f, indent=4)
    
print(f"✅ Ground truth manually corrected and successfully saved to {OUTPUT_FILE}!")

✅ Ground truth manually corrected and successfully saved to ../data/ground_truth.json!
